# 05 — News Analyst: Supervisor Multi-Agent Pattern

### Core concepts covered
| Concept | Description |
|---------|-------------|
| Supervisor pattern | One "manager" node decides which worker runs next |
| Workers report back | After each worker, execution returns to the supervisor |
| Conditional routing | `add_conditional_edges` with a router function |
| Tavily live search | Real web search integrated into the graph |

### Supervisor routing logic
```
no raw_news   →  fetcher    →  supervisor
no summary    →  summarizer →  supervisor
no analysis   →  analyst    →  supervisor
all done      →  compile_report → END
```

> **Before running:** set `GROQ_API_KEY` and `TAVILY_API_KEY` in a `.env` file.

## 1. Install Dependencies

In [ ]:
# !pip install langgraph langchain langchain-groq langchain-tavily python-dotenv

## 2. Imports & Setup

In [ ]:
import os
from typing import TypedDict, Literal
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()
print("GROQ_API_KEY  set:", bool(os.getenv("GROQ_API_KEY")))
print("TAVILY_API_KEY set:", bool(os.getenv("TAVILY_API_KEY")))

In [ ]:
llm         = init_chat_model("groq:llama3-8b-8192")
search_tool = TavilySearch(max_results=3)

## 3. Define the State

In [ ]:
class NewsState(TypedDict):
    topic:        str   # user-provided topic
    raw_news:     str   # filled by fetcher
    summary:      str   # filled by summarizer
    analysis:     str   # filled by analyst
    final_report: str   # compiled report
    next_agent:   str   # supervisor's routing decision

## 4. Supervisor Node

The supervisor reads the current state and decides which worker to call next.
It doesn't do any real work — it only **routes**.

In [ ]:
def supervisor(state: NewsState) -> dict:
    """Reads state flags and decides which worker runs next."""
    has_news     = bool(state.get("raw_news",  ""))
    has_summary  = bool(state.get("summary",   ""))
    has_analysis = bool(state.get("analysis",  ""))

    if not has_news:
        next_agent = "fetcher"
        msg = "📋 Supervisor → Fetcher : gathering news..."
    elif not has_summary:
        next_agent = "summarizer"
        msg = "📋 Supervisor → Summarizer : news ready, summarising..."
    elif not has_analysis:
        next_agent = "analyst"
        msg = "📋 Supervisor → Analyst : summary ready, analysing..."
    else:
        next_agent = "done"
        msg = "✅ Supervisor : all steps complete. Compiling report."

    print(f"\n  {msg}")
    return {"next_agent": next_agent}

## 5. Worker Agents

In [ ]:
def fetcher(state: NewsState) -> dict:
    """Worker 1 — search the web for recent news using Tavily."""
    print(f'  🌐 Fetcher : searching for "{state["topic"]}"...')
    try:
        results = search_tool.invoke(state["topic"])
        if isinstance(results, list):
            raw_news = "\n\n".join(
                f"[Source {i+1}]\n{r.get('content', str(r))[:600]}"
                for i, r in enumerate(results[:3])
            )
        else:
            raw_news = str(results)[:2000]
    except Exception as e:
        raw_news = (
            f"[Live search unavailable: {e}]\n"
            f'Placeholder: "{state["topic"]}" is a significant topic '
            f"with major recent developments."
        )
    print("  ✅ Fetcher done.")
    return {"raw_news": raw_news}

In [ ]:
def summarizer(state: NewsState) -> dict:
    """Worker 2 — condense the raw news into 3–4 sentences."""
    print("  📝 Summarizer : condensing news...")
    prompt = (
        f'Summarise the following news content about "{state["topic"]}" '
        f"in 3–4 clear sentences.\n"
        f"Focus on the most recent and important developments only.\n\n"
        f"News content:\n{state['raw_news'][:2500]}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print("  ✅ Summarizer done.")
    return {"summary": response.content.strip()}

In [ ]:
def analyst(state: NewsState) -> dict:
    """Worker 3 — extract sentiment, themes, and outlook."""
    print("  📊 Analyst : extracting insights...")
    prompt = (
        f'Analyse this news summary about "{state["topic"]}":\n\n'
        f"{state['summary']}\n\n"
        "Provide:\n"
        "1. Sentiment      : (Positive / Negative / Neutral / Mixed) — one sentence why\n"
        "2. Key Themes     : 3–5 bullet points of the main topics covered\n"
        "3. Outlook        : one sentence on what this signals going forward"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    print("  ✅ Analyst done.")
    return {"analysis": response.content.strip()}

In [ ]:
def compile_report(state: NewsState) -> dict:
    """Compile all outputs into a clean news intelligence brief."""
    report = (
        f"\n{'═'*55}\n"
        f"        📰  NEWS INTELLIGENCE REPORT\n"
        f"{'═'*55}\n"
        f"  Topic : {state['topic']}\n"
        f"{'═'*55}\n\n"
        f"📌  SUMMARY\n{'─'*45}\n"
        f"{state['summary']}\n\n"
        f"🔍  ANALYSIS & SENTIMENT\n{'─'*45}\n"
        f"{state['analysis']}\n\n"
        f"{'═'*55}"
    )
    print(report)
    return {"final_report": report}

## 6. Router Function

The router translates the supervisor's `next_agent` string into the actual
node name for `add_conditional_edges`.

In [ ]:
def router(state: NewsState) -> Literal["fetcher", "summarizer", "analyst", "compile_report"]:
    """Map supervisor's decision to a graph node name."""
    mapping = {
        "fetcher":    "fetcher",
        "summarizer": "summarizer",
        "analyst":    "analyst",
        "done":       "compile_report",
    }
    return mapping.get(state.get("next_agent", "done"), "compile_report")

## 7. Build & Compile the Graph

In [ ]:
builder = StateGraph(NewsState)

builder.add_node("supervisor",     supervisor)
builder.add_node("fetcher",        fetcher)
builder.add_node("summarizer",     summarizer)
builder.add_node("analyst",        analyst)
builder.add_node("compile_report", compile_report)

# Entry point
builder.add_edge(START, "supervisor")

# Supervisor uses conditional routing
builder.add_conditional_edges(
    "supervisor",
    router,
    {
        "fetcher":        "fetcher",
        "summarizer":     "summarizer",
        "analyst":        "analyst",
        "compile_report": "compile_report",
    },
)

# Every worker reports back to the supervisor
builder.add_edge("fetcher",        "supervisor")
builder.add_edge("summarizer",     "supervisor")
builder.add_edge("analyst",        "supervisor")
builder.add_edge("compile_report", END)

graph = builder.compile()
print("Graph compiled!")

## 8. Visualise the Graph *(optional)*

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Visualisation skipped: {e}")

## 9. Run the News Analyst

Enter any topic — the supervisor will orchestrate fetching, summarising, and analysing.

In [ ]:
topic = input("Enter a news topic (e.g., 'AI regulation', 'climate tech'): ").strip()
if not topic:
    topic = "artificial intelligence in 2025"

print(f"\n🚀 Analysing: {topic}\n")

graph.invoke({
    "topic":        topic,
    "raw_news":     "",
    "summary":      "",
    "analysis":     "",
    "final_report": "",
    "next_agent":   "",
})